https://learn.microsoft.com/en-us/azure/databricks/ldp/flows-backfill

**Backfill History tables based on year Partition**

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE retailraw.staging.stagingorderitems_raw_2025 AS
SELECT * FROM STREAM read_files(
  "/Volumes/hidi/raw/hidi/Retail/Orders/2025/*",
  format => "PARQUET",
  inferColumnTypes => true,
  maxFilesPerTrigger => 100,
  schemaEvolutionMode => "addNewColumns",
  modifiedAfter => "2024-12-31T23:59:59.999+00:00"
)
--WH

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE retailraw.staging.stagingorderitems_raw_2026 AS
SELECT * FROM STREAM read_files(
  "/Volumes/hidi/raw/hidi/Retail/Orders/2026/*",
  format => "PARQUET",
  inferColumnTypes => true,
  maxFilesPerTrigger => 100,
  schemaEvolutionMode => "addNewColumns",
  modifiedAfter => "2024-12-31T23:59:59.999+00:00"
)
--WH

In [0]:
CREATE  OR REFRESH STREAMING LIVE TABLE retailraw.staging.stagingorderitems_raw_all AS
SELECT * FROM STREAM read_files(
  "/Volumes/hidi/raw/hidi/Retail/Orders/*/*",
  format => "PARQUET",
  inferColumnTypes => true,
  maxFilesPerTrigger => 100,
  schemaEvolutionMode => "addNewColumns",
  modifiedAfter => "2024-12-31T23:59:59.999+00:00"
)

In [0]:
/*CREATE  OR REFRESH STREAMING  TABLE retailraw.staging.stagingorderitems_raw_all_Test AS
SELECT * FROM STREAM read_files(
  "/Volumes/hidi/raw/hidi/Retail/Orders/*/*",
  format => "PARQUET",
  inferColumnTypes => true,
  maxFilesPerTrigger => 100,
  schemaEvolutionMode => "addNewColumns",
  modifiedAfter => "2024-12-31T23:59:59.999+00:00"
)*/

In [0]:
SELECT COUNT(1) FROM retailraw.staging.stagingorderitems_raw_2026
union all
select count(1) from retailraw.staging.stagingorderitems_raw_2025  
union ALL
select count(1) from retailraw.staging.stagingorderitems_raw_all 

In [0]:
/*CREATE TABLE retailraw.staging.stagingorderitems_raw_2026 AS SELECT * FROM retailraw.staging.stagingorderitems_raw;*/

Mehtod 2 : Python method back fill load Hisorical **data**

In [0]:
%python
from pyspark import pipelines as dp

source_root_path = "/Volumes/hidi/raw/hidi/Retail/Orders/2026"
#begin_year = spark.conf.get("begin_year")
incremental_load_path = f"{source_root_path}/*"
incremental_load_path

In [0]:
%python
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp

source_root_path = "/Volumes/hidi/raw/hidi/Retail/Orders/"
#begin_year = spark.conf.get("begin_year")
incremental_load_path = f"{source_root_path}/*"

# create a streaming table and the default flow to ingest streaming events
@dp.table(name="retailraw.staging.stagingorderitems_raw_FullLoad", comment="Raw stagingorderitems")
def ingest():
    return (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.maxFilesPerTrigger", 100)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("modifiedAfter", "2025-01-01T00:00:00.000+00:00")
        .load(incremental_load_path)
        #.where(f"year(timestamp) >= {begin_year}") # safeguard to not process data before begin_year
    )

In [0]:
select * from retailraw.staging.stagingorderitems_raw_FullLoad limit 100

In [0]:
select count(1) from retailraw.staging.stagingorderitems_raw_FullLoad

In [0]:
drop table retailraw.staging.stagingorderitems_raw_FullLoadTest

Method 2: Python Method Backfill Hisory load with ETL_TTimestamp / Filename

In [0]:
%python
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.functions import (
    current_timestamp,
    input_file_name
)

source_root_path = "/Volumes/hidi/raw/hidi/Retail/Orders/"
incremental_load_path = f"{source_root_path}/*"

# Create streaming table
@dp.table(
    name="retailraw.staging.stagingorderitems_raw_FullLoadTest",
    comment="Raw stagingorderitems"
)
def ingest():

    return (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.maxFilesPerTrigger", 100)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("modifiedAfter", "2025-01-01T00:00:00.000+00:00")
        .load(incremental_load_path)

        # Custom metadata columns
        .withColumn("RawIngestionTimestamp", current_timestamp())
        .withColumn("RawSourceFileName", col("_metadata.file_path"))
    )

In [0]:
select count(1) as cnt,
RawSourceFileName
from retailraw.staging.stagingorderitems_raw_FullLoadTest
group by RawSourceFileName

Method 3: Load Backfill By each Year

In [0]:
%python
from pyspark.sql.functions import lit
from pyspark import pipelines as dp
job_id = "12345"
job_name = "BackFillLoads"
#source_root_path = spark.conf.get("registration_events_source_root_path")
source_root_path = "/Volumes/hidi/raw/hidi/Retail/Orders/"
begin_year = "2025"
#backfill_years = spark.conf.get("backfill_years") # e.g. "2024,2023,2022"
backfill_years  = "2025,2026"
incremental_load_path = f"{source_root_path}/*"

# NOTE: dp.create_streaming_table() and @dp.append_flow() only work inside a
# Lakeflow pipeline. In an interactive notebook only @dp.table is supported, so
# each year gets its own streaming table with a one-time batch load.

# meta programming to create a backfill table for a given year (called later)
def setup_backfill_table(year):
    backfill_path = f"{source_root_path}/{year}/*"
    @dp.table(
        name=f"stagingorderitems_raw_FullLoadByYear_{year}",
        comment=f"Backfill {year} raw stagingorderitems"
    )
    def backfill():
        return (
            spark
            .read
            .format("parquet")
            .load(backfill_path)
        )

# the original incremental, streaming load
@dp.table(
    name="stagingorderitems_raw_FullLoadByYear",
    comment="Raw stagingorderitems incremental load"
)
def ingest():
    return (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.maxFilesPerTrigger", 100)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("modifiedAfter", "2024-12-31T23:59:59.999+00:00")
        .load(incremental_load_path)
         # Custom metadata columns
        .withColumn("RawIngestionTimestamp", current_timestamp())
        .withColumn("RawSourceFileName", col("_metadata.file_path"))
        .withColumn("JobID", lit(job_id))
        .withColumn("JobName", lit(job_name))
        
    )

# parallelize one time multi years backfill for faster processing
# split backfill_years into array
for year in backfill_years.split(","):
    setup_backfill_table(year) # create the backfill table for each year

In [0]:
select count(1) from stagingorderitems_raw_FullLoadByYear;
drop table stagingorderitems_raw_FullLoadByYear

In [0]:

SELECT COUNT(1) AS CNT,
RawSourceFileName
from stagingorderitems_raw_FullLoadByYear
group by RawSourceFileName

In [0]:
select count(1) from retailraw.staging.stagingorderitems_raw_2026
UNION ALL
select count(1) from retailraw.staging.stagingorderitems_raw_2025;

**Method BACK FILL HISTORICAL SQL Methods**

In [0]:
-- create the streaming table
-- NOTE: standalone CREATE FLOW ... AS INSERT INTO only works inside a
-- Lakeflow pipeline. In a notebook, define streaming tables directly,
-- as in cells 2-4 above.

-- the original incremental, streaming load
CREATE OR REFRESH STREAMING LIVE TABLE retailraw.staging.stagingorderitems_raw_sql AS
SELECT * FROM STREAM read_files(
  "/Volumes/hidi/raw/hidi/Retail/Orders/*",
  format => "parquet",
  inferColumnTypes => true,
  maxFilesPerTrigger => 100,
  schemaEvolutionMode => "addNewColumns",
  modifiedAfter => "2024-12-31T23:59:59.999+00:00"
);
--WHERE year(timestamp) >= '2025'

-- one time backfill 2025
CREATE OR REFRESH STREAMING LIVE TABLE retailraw.staging.stagingorderitems_raw_sql_2025 AS
SELECT * FROM STREAM read_files(
  "/Volumes/hidi/raw/hidi/Retail/Orders/2025/*",
  format => "parquet",
  inferColumnTypes => true
);

-- one time backfill 2026
CREATE OR REFRESH STREAMING LIVE TABLE retailraw.staging.stagingorderitems_raw_sql_2026 AS
SELECT * FROM STREAM read_files(
  "/Volumes/hidi/raw/hidi/Retail/Orders/2026/*",
  format => "parquet",
  inferColumnTypes => true
);



In [0]:
drop table retailraw.staging.stagingorderitems_raw_sql;
drop table retailraw.staging.stagingorderitems_raw_sql_2025;
drop table retailraw.staging.stagingorderitems_raw_sql_2026

In [0]:
select * from retailraw.staging.stagingorderitems_raw_sql limit 10;

In [0]:
SELECT COUNT(1) FROM retailraw.staging.stagingorderitems_raw_sql
UNION ALL
SELECT COUNT(1) FROM retailraw.staging.stagingorderitems_raw_sql_2025
UNION ALL
SELECT COUNT(1) FROM retailraw.staging.stagingorderitems_raw_sql_2026